# Phase 12 Corrected — Two-Pass Inference with Paper-Accurate Baselines and Sampling Fusion

**Purpose**: Corrects the Phase 12 baseline computation by:
1. Running paper-accurate **Likelihood-Weighted Semantic Entropy** (Farquhar et al., Nature 2024) instead of D-SE (count-only).
2. Running **SelfCheckGPT official** (soft contradiction probability, correct premise=sentence ordering, auto-detected class index) instead of hard-argmax.
3. Keeping our spectral L-SML as a strict **1-pass** method (single `generate_full()` per question).
4. Enabling **sampling fusion** (advisor meeting Item 5): fuse LW-SE (K=10) with L-SML (1-pass) and measure AUROC gain.

**Branch**: `analysis/theorem-validation` (has Step 145 baselines + `lsml_continuous_pipeline`).

**Cache dir**: `phase12_corrected/` — separate from Phase 12's `phase12_baselines/`.

In [4]:
import os, sys, shutil

# Set BEFORE any torch import
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

REPO_DIR = '/content/hallucination_detection'
BRANCH   = 'master'

if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    shutil.rmtree(REPO_DIR)

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {BRANCH} https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull -q')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.system('pip install -q "transformers>=4.40" accelerate datasets bitsandbytes autoawq scipy tqdm')

from spectral_utils import (
    load_model, generate_full, free_memory,
    extract_all_features, FEAT_NAMES, load_cache, save_cache,
    boot_auc, lsml_continuous_pipeline,
    # Baselines -- legacy (D-SE + hard-argmax) for Phase 12 comparison row
    official_semantic_entropy, self_consistency_score, selfcheck_nli_score,
    # Baselines -- paper-accurate (Step 145)
    discrete_semantic_entropy, likelihood_weighted_semantic_entropy,
    selfcheck_nli_score_official,
    # Utilities
    nli_load_model, parse_verbalized_confidence, VERBALIZED_CONF_SUFFIX,
    # Dataset helpers
    load_gsm8k, gsm8k_prompt, is_correct_gsm8k, normalize_gsm8k, extract_model_answer_gsm8k,
    load_math500, math_prompt, is_correct_math,
    load_gpqa, gpqa_prompt_and_answer, is_correct_gpqa,
    load_lciteeval, lciteeval_prompt, lciteeval_grounding_label,
)
import scipy.stats
import pickle, os, numpy as np
from tqdm.auto import tqdm

# Freeze pyarrow BEFORE gptqmodel install (see CLAUDE.md)
import datasets  # noqa: F401

print('spectral_utils imported OK')
print(f'Branch: {BRANCH}')

spectral_utils imported OK
Branch: master


In [5]:
# ── Sampling config ────────────────────────────────────────────────────────
K_SE_SC      = 10      # K samples for SE + SC
K_CHECK      = 5       # K samples for SelfCheckGPT (first 5 of K_SE_SC)
TEMP         = 1.0
MAX_NEW_STD  = 1024    # Llama-8B, Qwen-7B-Instruct, GPQA (MCQ — short answers)
MAX_NEW_RSN  = 2048    # Qwen2.5-Math-7B (reasoning traces — longer than 1024)

# ── Subset sizes ──────────────────────────────────────────────────────────
N_MATH       = 200     # balanced 100 correct / 100 incorrect
N_GPQA       = 150     # balanced
N_RAG        = 200     # per RAG dataset

# ── NLI model ──────────────────────────────────────────────────────────────
NLI_MODEL    = 'cross-encoder/nli-deberta-v3-base'

# ── Cache dir — SEPARATE from Phase 12 to never overwrite ─────────────────
CACHE_DIR    = '/content/drive/MyDrive/hallucination_detection/cache/phase12_corrected'

# ── L-SML config (Step 134 GOOD_5 + continuous pipeline) ─────────────────
GOOD_FEATURES = ['epr', 'low_band_power', 'sw_var_peak', 'cusum_max', 'spectral_entropy']
FEATURE_SIGNS = {f: -1 for f in GOOD_FEATURES}  # higher entropy → more likely wrong
SE_SIGN       = 1                                 # higher SE → more likely hallucinated

# ── Incremental save helpers ───────────────────────────────────────────────
SAVE_EVERY = 25

def _save(cache, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(cache, f)

def _load_or_empty(path):
    if os.path.exists(path):
        return pickle.load(open(path, 'rb'))
    return {}

print('Config OK')

Config OK


In [6]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Cache dir: {CACHE_DIR}')

Mounted at /content/drive
Cache dir: /content/drive/MyDrive/hallucination_detection/cache/phase12_corrected


In [7]:
# Load NLI model to /content/nli_cache (SSD — faster than Drive for repeated NLI calls)
NLI_MDL, NLI_TOK, NLI_DEV = nli_load_model(
    model_name=NLI_MODEL, device='cuda', cache_dir='/content/nli_cache'
)
print(f'NLI loaded: {NLI_MODEL}')
print(f'NLI id2label: {NLI_MDL.config.id2label}')  # verify class order

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

NLI loaded: cross-encoder/nli-deberta-v3-base
NLI id2label: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}


## Section 1 — GSM8K / Llama-3.1-8B-Instruct

**Pass 1** (single inference): `token_entropies` → spectral features → L-SML.  
**Pass 2** (K=10 samples): `full_text` + `log_likelihood` per sample → LW-SE, D-SE, SC, SelfCheckGPT.

In [8]:
from huggingface_hub import snapshot_download

FLAT_CACHE = '/content/drive/MyDrive/hf_cache_flat'
os.makedirs(FLAT_CACHE, exist_ok=True)

def ensure_flat_dir(repo_id):
    local_dir = os.path.join(FLAT_CACHE, repo_id.replace('/', '__'))
    if os.path.exists(os.path.join(local_dir, 'config.json')):
        print(f'Using cached: {local_dir}')
        return local_dir
    print(f'Downloading {repo_id} to flat dir...')
    try:
        snapshot_download(repo_id=repo_id, local_dir=local_dir, local_dir_use_symlinks=False)
    except TypeError:
        snapshot_download(repo_id=repo_id, local_dir=local_dir)
    return local_dir

MDL_ID_GSM = 'meta-llama/Llama-3.1-8B-Instruct'
mdl, tok = load_model(ensure_flat_dir(MDL_ID_GSM), quantize_4bit=False)
print('Model loaded')


Using cached: /content/drive/MyDrive/hf_cache_flat/meta-llama__Llama-3.1-8B-Instruct


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded /content/drive/MyDrive/hf_cache_flat/meta-llama__Llama-3.1-8B-Instruct
Model loaded


In [9]:
P1_PATH = os.path.join(CACHE_DIR, 'p1_gsm8k_llama8b.pkl')
FORCE_RECOMPUTE = False

p1 = {} if FORCE_RECOMPUTE else _load_or_empty(P1_PATH)
print(f'Resuming: {sum(v.get("done") for v in p1.values())} already done')

gsm8k_rows = load_gsm8k()[:N_MATH]
print(f'Dataset: {len(gsm8k_rows)} samples')

for i, row in enumerate(tqdm(gsm8k_rows)):
    if p1.get(i, {}).get('done'):
        continue
    prompt = gsm8k_prompt(row['question'])

    # Pass 1 -- single inference for spectral features
    r1      = generate_full(mdl, tok, prompt, temperature=TEMP, max_new_tokens=MAX_NEW_STD)
    correct = int(is_correct_gsm8k(r1['full_text'], row['answer']))

    # Pass 2 -- K=10 samples for SE / SC / SelfCheckGPT
    k_samples = []
    for _ in range(K_SE_SC):
        rk = generate_full(mdl, tok, prompt, temperature=TEMP, max_new_tokens=MAX_NEW_STD)
        k_samples.append({
            'full_text':      rk['full_text'],
            # Length-normalized log-likelihood -- matches Farquhar et al. line 243
            'log_likelihood': float(np.mean(-np.array(rk['token_spilled_energies']))),
        })

    p1[i] = {
        'done':            True,
        'correct':         correct,
        'token_entropies': r1['token_entropies'],
        'token_spilled':   r1['token_spilled_energies'],
        'main_text':       r1['full_text'],
        'k_samples':       k_samples,
    }
    if (i + 1) % SAVE_EVERY == 0:
        _save(p1, P1_PATH)

_save(p1, P1_PATH)
n_done = sum(v.get('done') for v in p1.values())
print(f'Complete: {n_done} / {len(gsm8k_rows)}')

Resuming: 200 already done
Loaded 1319 GSM8K test problems.
Dataset: 200 samples


  0%|          | 0/200 [00:00<?, ?it/s]

Complete: 200 / 200


In [10]:
del mdl, tok
free_memory()
print('Llama-8B unloaded')

Llama-8B unloaded


In [11]:
# -- Valid entries --
p1 = _load_or_empty(P1_PATH)
valid1 = {k: v for k, v in p1.items() if v.get('done') and len(v['k_samples']) == K_SE_SC}
keys1  = sorted(valid1.keys())

# -- Spectral features (1-pass): extract once per key, drop too-short traces (None) --
_f1 = {k: extract_all_features(valid1[k]['token_entropies']) for k in keys1}
n_short1 = sum(f is None for f in _f1.values())
if n_short1:
    print(f'Dropped {n_short1} traces too short for spectral features')
keys1   = [k for k in keys1 if _f1[k] is not None]
labels1 = np.array([valid1[k]['correct'] for k in keys1])
print(f'Valid: {len(keys1)}, pos rate: {labels1.mean():.2f}')

feats1 = {feat: np.array([_f1[k][feat] for k in keys1]) for feat in GOOD_FEATURES}

lsml1_scores, _ = lsml_continuous_pipeline(feats1, GOOD_FEATURES, FEATURE_SIGNS)
lsml1_auc, lsml1_lo, lsml1_hi = boot_auc(labels1, lsml1_scores)

# -- K-sample NLI scores (expensive -- cache separately) --
P1_SE_PATH = os.path.join(CACHE_DIR, 'p1_gsm8k_se.pkl')
FORCE_SE = False
se1 = {} if FORCE_SE else _load_or_empty(P1_SE_PATH)

for k in tqdm(keys1, desc='SE/SC/SCGPT (GSM8K)'):
    if k in se1:
        continue
    texts   = [s['full_text'] for s in valid1[k]['k_samples']]
    lls     = [s['log_likelihood'] for s in valid1[k]['k_samples']]
    answers = [extract_model_answer_gsm8k(t) for t in texts]
    se1[k] = {
        'dse':  official_semantic_entropy(texts, NLI_MDL, NLI_TOK, NLI_DEV, NLI_MODEL),
        'lwse': likelihood_weighted_semantic_entropy(texts, lls, NLI_MDL, NLI_TOK, NLI_DEV, NLI_MODEL),
        'sc':   self_consistency_score(answers, normalize_fn=normalize_gsm8k),
        'scgpt_hard':     selfcheck_nli_score(
                              valid1[k]['main_text'], texts[:K_CHECK], NLI_MDL, NLI_TOK, NLI_DEV, NLI_MODEL),
        'scgpt_official': selfcheck_nli_score_official(
                              valid1[k]['main_text'], texts[:K_CHECK], NLI_MDL, NLI_TOK, NLI_DEV, NLI_MODEL),
    }
    if len(se1) % SAVE_EVERY == 0:
        _save(se1, P1_SE_PATH)
_save(se1, P1_SE_PATH)

# SE/SCGPT: higher = more hallucinated (label=0); negate so boot_auc sees higher=correct
# SC: higher = more consistent = more correct; no negation needed
dse1    = np.array([se1[k]['dse']   for k in keys1])
lwse1   = np.array([se1[k]['lwse']  for k in keys1])
sc1     = np.array([se1[k]['sc']    for k in keys1])
scgpt1h = np.array([se1[k]['scgpt_hard']     for k in keys1])
scgpt1o = np.array([se1[k]['scgpt_official'] for k in keys1])

rows1 = [
    ('L-SML 1-pass (GOOD_5)',           lsml1_auc, lsml1_lo,  lsml1_hi),
    ('D-SE K=10 (Phase 12 method)',     *boot_auc(labels1, -dse1)),
    ('LW-SE K=10 (paper-accurate)',     *boot_auc(labels1, -lwse1)),
    ('SC K=10',                         *boot_auc(labels1,  sc1)),
    ('SelfCheckGPT hard K=5',           *boot_auc(labels1, -scgpt1h)),
    ('SelfCheckGPT official K=5',       *boot_auc(labels1, -scgpt1o)),
]
print('\nGSM8K / Llama-3.1-8B-Instruct')
print(f'{"Method":<35} {"AUROC":>7} {"95% CI":>17}')
print('-' * 62)
for name, auc, lo, hi in rows1:
    print(f'{name:<35} {auc:.3f}  [{lo:.3f}, {hi:.3f}]')

Valid: 200, pos rate: 0.80


SE/SC/SCGPT (GSM8K):   0%|          | 0/200 [00:00<?, ?it/s]


GSM8K / Llama-3.1-8B-Instruct
Method                                AUROC            95% CI
--------------------------------------------------------------
L-SML 1-pass (GOOD_5)               0.754  [0.659, 0.843]
D-SE K=10 (Phase 12 method)         0.614  [0.510, 0.711]
LW-SE K=10 (paper-accurate)         0.613  [0.509, 0.709]
SC K=10                             0.608  [0.498, 0.713]
SelfCheckGPT hard K=5               0.601  [0.503, 0.706]
SelfCheckGPT official K=5           0.701  [0.611, 0.791]


In [12]:
# -- Sampling fusion: L-SML (1-pass) + LW-SE (K=10) --
rho1, _ = scipy.stats.spearmanr(-lsml1_scores, lwse1)
print(f'GSM8K  Spearman rho(L-SML, LW-SE) = {rho1:.3f}  '
      f'({"OK to fuse" if abs(rho1) < 0.75 else "WARNING: high correlation"})')

feats1_fused = {**feats1, 'lwse': lwse1}
signs1_fused = {**FEATURE_SIGNS, 'lwse': SE_SIGN}
fused1_scores, _ = lsml_continuous_pipeline(feats1_fused, GOOD_FEATURES + ['lwse'], signs1_fused)
fused1_auc, fused1_lo, fused1_hi = boot_auc(labels1, fused1_scores)

lwse1_auc = boot_auc(labels1, -lwse1)[0]
print(f'Fusion (L-SML + LW-SE): {fused1_auc:.3f} [{fused1_lo:.3f}, {fused1_hi:.3f}]')
print(f'  vs L-SML alone: {lsml1_auc:.3f}  |  LW-SE alone: {lwse1_auc:.3f}')
print(f'  Fusion gain vs L-SML: {fused1_auc - lsml1_auc:+.3f}')
print(f'  Fusion gain vs LW-SE: {fused1_auc - lwse1_auc:+.3f}')

GSM8K  Spearman rho(L-SML, LW-SE) = 0.263  (OK to fuse)
Fusion (L-SML + LW-SE): 0.758 [0.662, 0.845]
  vs L-SML alone: 0.754  |  LW-SE alone: 0.613
  Fusion gain vs L-SML: +0.004
  Fusion gain vs LW-SE: +0.145


## Section 2 — MATH-500 / Qwen2.5-Math-7B-Instruct

Same two-pass pattern. `MAX_NEW_RSN=2048` for both passes (reasoning traces are longer).

In [13]:
MDL_ID_MATH = 'Qwen/Qwen2.5-Math-7B-Instruct'
mdl, tok = load_model(ensure_flat_dir(MDL_ID_MATH), quantize_4bit=False)
print(f'Model loaded: {MDL_ID_MATH}')

Using cached: /content/drive/MyDrive/hf_cache_flat/Qwen__Qwen2.5-Math-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded /content/drive/MyDrive/hf_cache_flat/Qwen__Qwen2.5-Math-7B-Instruct
Model loaded: Qwen/Qwen2.5-Math-7B-Instruct


In [14]:
P2_PATH = os.path.join(CACHE_DIR, 'p2_math500_qwenmath7b.pkl')
FORCE_RECOMPUTE = False

p2 = {} if FORCE_RECOMPUTE else _load_or_empty(P2_PATH)
print(f'Resuming: {sum(v.get("done") for v in p2.values())} already done')

math_rows = load_math500(n_samples=N_MATH)
print(f'Dataset: {len(math_rows)} samples')

import re as _re_math
def _extract_boxed(t):
    m = _re_math.search(r'\\boxed\{([^}]*)\}', t)
    return m.group(1).strip() if m else t.strip()[-30:]

for i, row in enumerate(tqdm(math_rows)):
    if p2.get(i, {}).get('done'):
        continue
    prompt = math_prompt(row)

    # Pass 1 -- single inference for spectral features
    r1      = generate_full(mdl, tok, prompt, temperature=TEMP, max_new_tokens=MAX_NEW_RSN)
    correct = int(is_correct_math(r1['full_text'], row))

    # Pass 2 -- K=10 samples
    k_samples = []
    for _ in range(K_SE_SC):
        rk = generate_full(mdl, tok, prompt, temperature=TEMP, max_new_tokens=MAX_NEW_RSN)
        k_samples.append({
            'full_text':      rk['full_text'],
            'log_likelihood': float(np.mean(-np.array(rk['token_spilled_energies']))),
        })

    p2[i] = {
        'done':            True,
        'correct':         correct,
        'token_entropies': r1['token_entropies'],
        'token_spilled':   r1['token_spilled_energies'],
        'main_text':       r1['full_text'],
        'k_samples':       k_samples,
    }
    if (i + 1) % SAVE_EVERY == 0:
        _save(p2, P2_PATH)

_save(p2, P2_PATH)
print(f'Complete: {sum(v.get("done") for v in p2.values())} / {len(math_rows)}')

Resuming: 50 already done
  lighteval/MATH_500 failed: Dataset 'lighteval/MATH_500' doesn't exist on the Hub or cannot be accessed.
Loaded 200 MATH problems from HuggingFaceH4/MATH-500.
Dataset: 200 samples


  0%|          | 0/200 [00:00<?, ?it/s]

Complete: 200 / 200


In [15]:
del mdl, tok
free_memory()
print('Qwen-Math-7B unloaded')

Qwen-Math-7B unloaded


In [16]:
p2 = _load_or_empty(P2_PATH)
valid2  = {k: v for k, v in p2.items() if v.get('done') and len(v['k_samples']) == K_SE_SC}
keys2   = sorted(valid2.keys())

# Spectral features: extract once per key, drop too-short traces (None)
_f2 = {k: extract_all_features(valid2[k]['token_entropies']) for k in keys2}
n_short2 = sum(f is None for f in _f2.values())
if n_short2:
    print(f'Dropped {n_short2} traces too short for spectral features')
keys2   = [k for k in keys2 if _f2[k] is not None]
labels2 = np.array([valid2[k]['correct'] for k in keys2])
print(f'Valid: {len(keys2)}, pos rate: {labels2.mean():.2f}')

feats2 = {feat: np.array([_f2[k][feat] for k in keys2]) for feat in GOOD_FEATURES}
lsml2_scores, _ = lsml_continuous_pipeline(feats2, GOOD_FEATURES, FEATURE_SIGNS)
lsml2_auc, lsml2_lo, lsml2_hi = boot_auc(labels2, lsml2_scores)

# K-sample NLI scores
P2_SE_PATH = os.path.join(CACHE_DIR, 'p2_math500_se.pkl')
FORCE_SE = False
se2 = {} if FORCE_SE else _load_or_empty(P2_SE_PATH)

for k in tqdm(keys2, desc='SE/SC/SCGPT (MATH-500)'):
    if k in se2:
        continue
    texts = [s['full_text'] for s in valid2[k]['k_samples']]
    lls   = [s['log_likelihood'] for s in valid2[k]['k_samples']]
    answers = [_extract_boxed(t) for t in texts]
    se2[k] = {
        'dse':  official_semantic_entropy(texts, NLI_MDL, NLI_TOK, NLI_DEV, NLI_MODEL),
        'lwse': likelihood_weighted_semantic_entropy(texts, lls, NLI_MDL, NLI_TOK, NLI_DEV, NLI_MODEL),
        'sc':   self_consistency_score(answers),
        'scgpt_hard':     selfcheck_nli_score(
                              valid2[k]['main_text'], texts[:K_CHECK], NLI_MDL, NLI_TOK, NLI_DEV, NLI_MODEL),
        'scgpt_official': selfcheck_nli_score_official(
                              valid2[k]['main_text'], texts[:K_CHECK], NLI_MDL, NLI_TOK, NLI_DEV, NLI_MODEL),
    }
    if len(se2) % SAVE_EVERY == 0:
        _save(se2, P2_SE_PATH)
_save(se2, P2_SE_PATH)

dse2    = np.array([se2[k]['dse']   for k in keys2])
lwse2   = np.array([se2[k]['lwse']  for k in keys2])
sc2     = np.array([se2[k]['sc']    for k in keys2])
scgpt2h = np.array([se2[k]['scgpt_hard']     for k in keys2])
scgpt2o = np.array([se2[k]['scgpt_official'] for k in keys2])

rows2 = [
    ('L-SML 1-pass (GOOD_5)',           lsml2_auc, lsml2_lo,  lsml2_hi),
    ('D-SE K=10 (Phase 12 method)',     *boot_auc(labels2, -dse2)),
    ('LW-SE K=10 (paper-accurate)',     *boot_auc(labels2, -lwse2)),
    ('SC K=10',                         *boot_auc(labels2,  sc2)),
    ('SelfCheckGPT hard K=5',           *boot_auc(labels2, -scgpt2h)),
    ('SelfCheckGPT official K=5',       *boot_auc(labels2, -scgpt2o)),
]
print('\nMATH-500 / Qwen2.5-Math-7B-Instruct')
print(f'{"Method":<35} {"AUROC":>7} {"95% CI":>17}')
print('-' * 62)
for name, auc, lo, hi in rows2:
    print(f'{name:<35} {auc:.3f}  [{lo:.3f}, {hi:.3f}]')

# Sampling fusion
rho2, _ = scipy.stats.spearmanr(-lsml2_scores, lwse2)
print(f'\nMATH-500  Spearman rho(L-SML, LW-SE) = {rho2:.3f}  '
      f'({"OK" if abs(rho2) < 0.75 else "WARN: high correlation"})')
feats2_fused  = {**feats2, 'lwse': lwse2}
signs2_fused  = {**FEATURE_SIGNS, 'lwse': SE_SIGN}
fused2_scores, _ = lsml_continuous_pipeline(feats2_fused, GOOD_FEATURES + ['lwse'], signs2_fused)
fused2_auc, fused2_lo, fused2_hi = boot_auc(labels2, fused2_scores)
lwse2_auc = boot_auc(labels2, -lwse2)[0]
print(f'Fusion (L-SML + LW-SE): {fused2_auc:.3f} [{fused2_lo:.3f}, {fused2_hi:.3f}]')
print(f'  vs L-SML: {lsml2_auc:.3f}  |  LW-SE: {lwse2_auc:.3f}')
print(f'  Gain vs L-SML: {fused2_auc - lsml2_auc:+.3f}  |  vs LW-SE: {fused2_auc - lwse2_auc:+.3f}')

Valid: 200, pos rate: 0.69


SE/SC/SCGPT (MATH-500):   0%|          | 0/200 [00:00<?, ?it/s]


MATH-500 / Qwen2.5-Math-7B-Instruct
Method                                AUROC            95% CI
--------------------------------------------------------------
L-SML 1-pass (GOOD_5)               0.230  [0.152, 0.316]
D-SE K=10 (Phase 12 method)         0.630  [0.552, 0.708]
LW-SE K=10 (paper-accurate)         0.625  [0.545, 0.703]
SC K=10                             0.863  [0.796, 0.913]
SelfCheckGPT hard K=5               0.549  [0.471, 0.627]
SelfCheckGPT official K=5           0.593  [0.506, 0.673]

MATH-500  Spearman rho(L-SML, LW-SE) = -0.251  (OK)
Fusion (L-SML + LW-SE): 0.232 [0.150, 0.322]
  vs L-SML: 0.230  |  LW-SE: 0.625
  Gain vs L-SML: +0.002  |  vs LW-SE: -0.394


## Section 3 — GPQA Diamond / Qwen2.5-7B-Instruct

MCQ format (A/B/C/D). SC uses majority-vote on extracted letter. VC is a separate 0-temperature pass.

In [17]:
MDL_ID_GPQA = 'Qwen/Qwen2.5-7B-Instruct'
mdl, tok = load_model(ensure_flat_dir(MDL_ID_GPQA), quantize_4bit=False)
print(f'Model loaded: {MDL_ID_GPQA}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded /content/drive/MyDrive/hf_cache_flat/Qwen__Qwen2.5-7B-Instruct
Model loaded: Qwen/Qwen2.5-7B-Instruct


In [18]:
P3_PATH    = os.path.join(CACHE_DIR, 'p3_gpqa_qwen7b.pkl')
P3_VC_PATH = os.path.join(CACHE_DIR, 'p3_gpqa_qwen7b_vc.pkl')
FORCE_RECOMPUTE = False

p3    = {} if FORCE_RECOMPUTE else _load_or_empty(P3_PATH)
p3_vc = _load_or_empty(P3_VC_PATH)
print(f'Resuming inference: {sum(v.get("done") for v in p3.values())} done')
print(f'Resuming VC: {sum(v.get("done") for v in p3_vc.values())} done')

gpqa_rows = load_gpqa()[:N_GPQA]
print(f'Dataset: {len(gpqa_rows)} samples')

import re as _re
def _extract_mcq(text):
    m = _re.search(r'\b([A-D])\b', text.upper())
    return m.group(1) if m else ''

for i, row in enumerate(tqdm(gpqa_rows)):
    prompt, gold = gpqa_prompt_and_answer(row, i)

    # -- Main inference (Pass 1) + K-samples (Pass 2) --
    if not p3.get(i, {}).get('done'):
        r1      = generate_full(mdl, tok, prompt, temperature=TEMP, max_new_tokens=MAX_NEW_STD)
        correct = int(is_correct_gpqa(r1['full_text'], gold))

        k_samples = []
        for _ in range(K_SE_SC):
            rk = generate_full(mdl, tok, prompt, temperature=TEMP, max_new_tokens=MAX_NEW_STD)
            k_samples.append({
                'full_text':      rk['full_text'],
                'log_likelihood': float(np.mean(-np.array(rk['token_spilled_energies']))),
                'answer':         _extract_mcq(rk['full_text']),
            })

        p3[i] = {
            'done':            True,
            'correct':         correct,
            'token_entropies': r1['token_entropies'],
            'token_spilled':   r1['token_spilled_energies'],
            'main_text':       r1['full_text'],
            'k_samples':       k_samples,
        }
        if (i + 1) % SAVE_EVERY == 0:
            _save(p3, P3_PATH)

    # -- Verbalized confidence (0-temperature, separate cache) --
    if not p3_vc.get(i, {}).get('done') and p3.get(i, {}).get('done'):
        vc_prompt = prompt + '\n\n' + p3[i]['main_text'] + VERBALIZED_CONF_SUFFIX
        vc_r = generate_full(mdl, tok, vc_prompt, temperature=0.0, max_new_tokens=20)
        p3_vc[i] = {
            'done': True,
            'conf': parse_verbalized_confidence(vc_r['full_text']),
        }
        if (i + 1) % SAVE_EVERY == 0:
            _save(p3_vc, P3_VC_PATH)

_save(p3, P3_PATH)
_save(p3_vc, P3_VC_PATH)
print(f'Inference: {sum(v.get("done") for v in p3.values())} / {len(gpqa_rows)}')
print(f'VC:        {sum(v.get("done") for v in p3_vc.values())} / {len(gpqa_rows)}')

Resuming inference: 0 done
Resuming VC: 0 done
Loaded 198 GPQA Diamond problems.
Dataset: 150 samples


  0%|          | 0/150 [00:00<?, ?it/s]

Inference: 150 / 150
VC:        150 / 150


In [19]:
del mdl, tok
free_memory()
print('Qwen-7B unloaded')

Qwen-7B unloaded


In [20]:
p3    = _load_or_empty(P3_PATH)
p3_vc = _load_or_empty(P3_VC_PATH)
valid3  = {k: v for k, v in p3.items() if v.get('done') and len(v['k_samples']) == K_SE_SC}
keys3   = sorted(valid3.keys())

# Spectral: extract once per key, drop too-short traces (None)
_f3 = {k: extract_all_features(valid3[k]['token_entropies']) for k in keys3}
n_short3 = sum(f is None for f in _f3.values())
if n_short3:
    print(f'Dropped {n_short3} traces too short for spectral features')
keys3   = [k for k in keys3 if _f3[k] is not None]
labels3 = np.array([valid3[k]['correct'] for k in keys3])
print(f'Valid: {len(keys3)}, pos rate: {labels3.mean():.2f}')

feats3 = {feat: np.array([_f3[k][feat] for k in keys3]) for feat in GOOD_FEATURES}
lsml3_scores, _ = lsml_continuous_pipeline(feats3, GOOD_FEATURES, FEATURE_SIGNS)
lsml3_auc, lsml3_lo, lsml3_hi = boot_auc(labels3, lsml3_scores)

# VC (higher confidence = more correct -- no negation needed)
vc3 = np.array([p3_vc.get(k, {}).get('conf', float('nan')) for k in keys3])
vc3_mask = ~np.isnan(vc3)
vc3_auc, vc3_lo, vc3_hi = boot_auc(labels3[vc3_mask], vc3[vc3_mask]) if vc3_mask.sum() > 10 else (float('nan'),)*3

# K-sample NLI scores
P3_SE_PATH = os.path.join(CACHE_DIR, 'p3_gpqa_se.pkl')
FORCE_SE = False
se3 = {} if FORCE_SE else _load_or_empty(P3_SE_PATH)

for k in tqdm(keys3, desc='SE/SC/SCGPT (GPQA)'):
    if k in se3:
        continue
    texts   = [s['full_text'] for s in valid3[k]['k_samples']]
    lls     = [s['log_likelihood'] for s in valid3[k]['k_samples']]
    answers = [s.get('answer', '') for s in valid3[k]['k_samples']]
    se3[k] = {
        'dse':  official_semantic_entropy(texts, NLI_MDL, NLI_TOK, NLI_DEV, NLI_MODEL),
        'lwse': likelihood_weighted_semantic_entropy(texts, lls, NLI_MDL, NLI_TOK, NLI_DEV, NLI_MODEL),
        'sc':   self_consistency_score(answers, normalize_fn=str.upper),
        'scgpt_hard':     selfcheck_nli_score(
                              valid3[k]['main_text'], texts[:K_CHECK], NLI_MDL, NLI_TOK, NLI_DEV, NLI_MODEL),
        'scgpt_official': selfcheck_nli_score_official(
                              valid3[k]['main_text'], texts[:K_CHECK], NLI_MDL, NLI_TOK, NLI_DEV, NLI_MODEL),
    }
    if len(se3) % SAVE_EVERY == 0:
        _save(se3, P3_SE_PATH)
_save(se3, P3_SE_PATH)

dse3    = np.array([se3[k]['dse']   for k in keys3])
lwse3   = np.array([se3[k]['lwse']  for k in keys3])
sc3     = np.array([se3[k]['sc']    for k in keys3])
scgpt3h = np.array([se3[k]['scgpt_hard']     for k in keys3])
scgpt3o = np.array([se3[k]['scgpt_official'] for k in keys3])

rows3 = [
    ('L-SML 1-pass (GOOD_5)',           lsml3_auc, lsml3_lo,  lsml3_hi),
    ('VC K=1',                          vc3_auc,   vc3_lo,    vc3_hi),
    ('D-SE K=10 (Phase 12 method)',     *boot_auc(labels3, -dse3)),
    ('LW-SE K=10 (paper-accurate)',     *boot_auc(labels3, -lwse3)),
    ('SC K=10',                         *boot_auc(labels3,  sc3)),
    ('SelfCheckGPT hard K=5',           *boot_auc(labels3, -scgpt3h)),
    ('SelfCheckGPT official K=5',       *boot_auc(labels3, -scgpt3o)),
]
print('\nGPQA Diamond / Qwen2.5-7B-Instruct')
print(f'{"Method":<35} {"AUROC":>7} {"95% CI":>17}')
print('-' * 62)
for name, auc, lo, hi in rows3:
    print(f'{name:<35} {auc:.3f}  [{lo:.3f}, {hi:.3f}]')

# Sampling fusion
rho3, _ = scipy.stats.spearmanr(-lsml3_scores, lwse3)
print(f'\nGPQA  Spearman rho(L-SML, LW-SE) = {rho3:.3f}  '
      f'({"OK" if abs(rho3) < 0.75 else "WARN: high correlation"})')
feats3_fused  = {**feats3, 'lwse': lwse3}
signs3_fused  = {**FEATURE_SIGNS, 'lwse': SE_SIGN}
fused3_scores, _ = lsml_continuous_pipeline(feats3_fused, GOOD_FEATURES + ['lwse'], signs3_fused)
fused3_auc, fused3_lo, fused3_hi = boot_auc(labels3, fused3_scores)
lwse3_auc = boot_auc(labels3, -lwse3)[0]
print(f'Fusion (L-SML + LW-SE): {fused3_auc:.3f} [{fused3_lo:.3f}, {fused3_hi:.3f}]')
print(f'  vs L-SML: {lsml3_auc:.3f}  |  LW-SE: {lwse3_auc:.3f}')
print(f'  Gain vs L-SML: {fused3_auc - lsml3_auc:+.3f}  |  vs LW-SE: {fused3_auc - lwse3_auc:+.3f}')

Valid: 150, pos rate: 0.31


SE/SC/SCGPT (GPQA):   0%|          | 0/150 [00:00<?, ?it/s]


GPQA Diamond / Qwen2.5-7B-Instruct
Method                                AUROC            95% CI
--------------------------------------------------------------
L-SML 1-pass (GOOD_5)               0.553  [0.445, 0.656]
VC K=1                              0.428  [0.334, 0.523]
D-SE K=10 (Phase 12 method)         0.504  [0.396, 0.601]
LW-SE K=10 (paper-accurate)         0.501  [0.391, 0.605]
SC K=10                             0.504  [0.410, 0.595]
SelfCheckGPT hard K=5               0.502  [0.400, 0.602]
SelfCheckGPT official K=5           0.512  [0.414, 0.613]

GPQA  Spearman rho(L-SML, LW-SE) = -0.188  (OK)
Fusion (L-SML + LW-SE): 0.573 [0.469, 0.670]
  vs L-SML: 0.553  |  LW-SE: 0.501
  Gain vs L-SML: +0.020  |  vs LW-SE: +0.072


## Section 4 — RAG / Qwen2.5-7B-Instruct × 4 Datasets

SelfCheckGPT only (hard vs official variant). No SE/SC — RAG outputs are retrieval-grounded and
don't benefit from clustering-based semantic uncertainty. Logprobs not needed for SelfCheckGPT.

In [21]:
# Reuse Qwen-7B-Instruct (same model as GPQA -- flat dir already cached)
MDL_ID_RAG = 'Qwen/Qwen2.5-7B-Instruct'
mdl, tok = load_model(ensure_flat_dir(MDL_ID_RAG), quantize_4bit=False)
print(f'Model loaded: {MDL_ID_RAG}')

Using cached: /content/drive/MyDrive/hf_cache_flat/Qwen__Qwen2.5-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded /content/drive/MyDrive/hf_cache_flat/Qwen__Qwen2.5-7B-Instruct
Model loaded: Qwen/Qwen2.5-7B-Instruct


In [22]:
RAG_DATASETS = ['hotpotqa', 'natural_questions', '2wikimultihopqa', 'narrativeqa']
rag_results = {}

import re as _re_cite_mod
_re_cite = _re_cite_mod.compile(r'\[(\d+)\]')

for ds_name in RAG_DATASETS:
    P4_PATH = os.path.join(CACHE_DIR, f'p4_rag_{ds_name}_qwen7b.pkl')
    FORCE_RECOMPUTE = False
    p4 = {} if FORCE_RECOMPUTE else _load_or_empty(P4_PATH)
    print(f'\n[{ds_name}] Resuming: {sum(v.get("done") for v in p4.values())} done')

    rag_rows = load_lciteeval(task=ds_name, n_samples=N_RAG)

    for i, row in enumerate(tqdm(rag_rows, desc=ds_name)):
        if p4.get(i, {}).get('done'):
            continue
        prompt = lciteeval_prompt(row)

        # Main response
        r1 = generate_full(mdl, tok, prompt, temperature=TEMP, max_new_tokens=MAX_NEW_STD)

        # Grounding label from the model's citation markers [n] in the response
        citation_ids = [int(m) for m in _re_cite.findall(r1['full_text'])]
        correct = int(lciteeval_grounding_label(citation_ids, row))

        # K=5 samples for SelfCheckGPT (text only -- no logprobs needed)
        samples = []
        for _ in range(K_CHECK):
            rk = generate_full(mdl, tok, prompt, temperature=TEMP, max_new_tokens=MAX_NEW_STD)
            samples.append(rk['full_text'])

        # Compute scores inline
        scgpt_h = selfcheck_nli_score(
            r1['full_text'], samples, NLI_MDL, NLI_TOK, NLI_DEV, NLI_MODEL)
        scgpt_o = selfcheck_nli_score_official(
            r1['full_text'], samples, NLI_MDL, NLI_TOK, NLI_DEV, NLI_MODEL)

        p4[i] = {
            'done':           True,
            'correct':        correct,
            'scgpt_hard':     scgpt_h,
            'scgpt_official': scgpt_o,
        }
        if (i + 1) % SAVE_EVERY == 0:
            _save(p4, P4_PATH)

    _save(p4, P4_PATH)

    # Compute AUROCs (negate: higher SCGPT score = more hallucinated = label=0)
    valid4  = {k: v for k, v in p4.items() if v.get('done')}
    keys4   = sorted(valid4.keys())
    labels4 = np.array([valid4[k]['correct'] for k in keys4])
    scgpt4h = np.array([valid4[k]['scgpt_hard']     for k in keys4])
    scgpt4o = np.array([valid4[k]['scgpt_official'] for k in keys4])
    auc_h, lo_h, hi_h = boot_auc(labels4, -scgpt4h)
    auc_o, lo_o, hi_o = boot_auc(labels4, -scgpt4o)
    rag_results[ds_name] = {'hard': (auc_h, lo_h, hi_h), 'official': (auc_o, lo_o, hi_o)}
    print(f'  SelfCheckGPT hard:     {auc_h:.3f} [{lo_h:.3f}, {hi_h:.3f}]')
    print(f'  SelfCheckGPT official: {auc_o:.3f} [{lo_o:.3f}, {hi_o:.3f}]')

del mdl, tok
free_memory()
print('\nQwen-7B unloaded')


[hotpotqa] Resuming: 0 done
Loaded 200 L-CiteEval samples (hotpotqa, config=L-CiteEval-Data_hotpotqa).


hotpotqa:   0%|          | 0/200 [00:00<?, ?it/s]

  SelfCheckGPT hard:     0.317 [0.147, 0.488]
  SelfCheckGPT official: 0.243 [0.137, 0.357]

[natural_questions] Resuming: 0 done
Loaded 160 L-CiteEval samples (natural_questions, config=L-CiteEval-Data_natural_questions).


natural_questions:   0%|          | 0/160 [00:00<?, ?it/s]

  SelfCheckGPT hard:     0.393 [0.206, 0.593]
  SelfCheckGPT official: 0.322 [0.122, 0.570]

[2wikimultihopqa] Resuming: 0 done
Loaded 200 L-CiteEval samples (2wikimultihopqa, config=L-CiteEval-Data_2wikimultihopqa).


2wikimultihopqa:   0%|          | 0/200 [00:00<?, ?it/s]

  SelfCheckGPT hard:     0.354 [0.168, 0.523]
  SelfCheckGPT official: 0.306 [0.116, 0.538]

[narrativeqa] Resuming: 0 done
Loaded 200 L-CiteEval samples (narrativeqa, config=L-CiteEval-Data_narrativeqa).


narrativeqa:   0%|          | 0/200 [00:00<?, ?it/s]

  SelfCheckGPT hard:     0.477 [0.333, 0.602]
  SelfCheckGPT official: 0.442 [0.313, 0.568]

Qwen-7B unloaded


## Section 5 — Summary

In [23]:
print('=' * 80)
print('PHASE 12 CORRECTED -- MASTER COMPARISON TABLE')
print('=' * 80)

def _row(domain, model, method, passes, auc, lo, hi):
    ci = f'[{lo:.3f}, {hi:.3f}]' if not (isinstance(auc, float) and np.isnan(auc)) else 'N/A'
    auc_s = f'{auc:.3f}' if not (isinstance(auc, float) and np.isnan(auc)) else 'N/A'
    print(f'{domain:<12} {model:<22} {method:<30} {passes:<6} {auc_s:>7}  {ci}')

print(f'{"Domain":<12} {"Model":<22} {"Method":<30} {"Pass":<6} {"AUROC":>7}  {"95% CI"}')
print('-' * 88)

# GSM8K
_row('GSM8K', 'Llama-3.1-8B', 'L-SML 1-pass',          '1',    lsml1_auc, lsml1_lo, lsml1_hi)
_row('GSM8K', 'Llama-3.1-8B', 'D-SE K=10 (Ph12)',       'K=10', *boot_auc(labels1, -dse1))
_row('GSM8K', 'Llama-3.1-8B', 'LW-SE K=10 (paper)',     'K=10', *boot_auc(labels1, -lwse1))
_row('GSM8K', 'Llama-3.1-8B', 'SC K=10',                'K=10', *boot_auc(labels1,  sc1))
_row('GSM8K', 'Llama-3.1-8B', 'SelfCheckGPT hard K=5',  'K=5',  *boot_auc(labels1, -scgpt1h))
_row('GSM8K', 'Llama-3.1-8B', 'SelfCheckGPT off. K=5',  'K=5',  *boot_auc(labels1, -scgpt1o))
_row('GSM8K', 'Llama-3.1-8B', 'Fusion (L-SML+LW-SE)',   '1+K',  fused1_auc, fused1_lo, fused1_hi)
print()

# MATH-500
_row('MATH-500', 'Qwen-Math-7B', 'L-SML 1-pass',          '1',    lsml2_auc, lsml2_lo, lsml2_hi)
_row('MATH-500', 'Qwen-Math-7B', 'D-SE K=10 (Ph12)',       'K=10', *boot_auc(labels2, -dse2))
_row('MATH-500', 'Qwen-Math-7B', 'LW-SE K=10 (paper)',     'K=10', *boot_auc(labels2, -lwse2))
_row('MATH-500', 'Qwen-Math-7B', 'SC K=10',                'K=10', *boot_auc(labels2,  sc2))
_row('MATH-500', 'Qwen-Math-7B', 'SelfCheckGPT hard K=5',  'K=5',  *boot_auc(labels2, -scgpt2h))
_row('MATH-500', 'Qwen-Math-7B', 'SelfCheckGPT off. K=5',  'K=5',  *boot_auc(labels2, -scgpt2o))
_row('MATH-500', 'Qwen-Math-7B', 'Fusion (L-SML+LW-SE)',   '1+K',  fused2_auc, fused2_lo, fused2_hi)
print()

# GPQA
_row('GPQA', 'Qwen2.5-7B', 'L-SML 1-pass',          '1',    lsml3_auc, lsml3_lo, lsml3_hi)
_row('GPQA', 'Qwen2.5-7B', 'VC K=1',                 '1',    vc3_auc,   vc3_lo,   vc3_hi)
_row('GPQA', 'Qwen2.5-7B', 'D-SE K=10 (Ph12)',        'K=10', *boot_auc(labels3, -dse3))
_row('GPQA', 'Qwen2.5-7B', 'LW-SE K=10 (paper)',      'K=10', *boot_auc(labels3, -lwse3))
_row('GPQA', 'Qwen2.5-7B', 'SC K=10',                 'K=10', *boot_auc(labels3,  sc3))
_row('GPQA', 'Qwen2.5-7B', 'SelfCheckGPT hard K=5',   'K=5',  *boot_auc(labels3, -scgpt3h))
_row('GPQA', 'Qwen2.5-7B', 'SelfCheckGPT off. K=5',   'K=5',  *boot_auc(labels3, -scgpt3o))
_row('GPQA', 'Qwen2.5-7B', 'Fusion (L-SML+LW-SE)',    '1+K',  fused3_auc, fused3_lo, fused3_hi)
print()

# RAG (AUROCs already negated at compute time in RAG cell)
for ds, res in rag_results.items():
    _row(f'RAG/{ds[:8]}', 'Qwen2.5-7B', 'SelfCheckGPT hard K=5',  'K=5', *res['hard'])
    _row(f'RAG/{ds[:8]}', 'Qwen2.5-7B', 'SelfCheckGPT off. K=5',  'K=5', *res['official'])

PHASE 12 CORRECTED -- MASTER COMPARISON TABLE
Domain       Model                  Method                         Pass     AUROC  95% CI
----------------------------------------------------------------------------------------
GSM8K        Llama-3.1-8B           L-SML 1-pass                   1        0.754  [0.659, 0.843]
GSM8K        Llama-3.1-8B           D-SE K=10 (Ph12)               K=10     0.614  [0.510, 0.711]
GSM8K        Llama-3.1-8B           LW-SE K=10 (paper)             K=10     0.613  [0.509, 0.709]
GSM8K        Llama-3.1-8B           SC K=10                        K=10     0.608  [0.498, 0.713]
GSM8K        Llama-3.1-8B           SelfCheckGPT hard K=5          K=5      0.601  [0.503, 0.706]
GSM8K        Llama-3.1-8B           SelfCheckGPT off. K=5          K=5      0.701  [0.611, 0.791]
GSM8K        Llama-3.1-8B           Fusion (L-SML+LW-SE)           1+K      0.758  [0.662, 0.845]

MATH-500     Qwen-Math-7B           L-SML 1-pass                   1        0.230  [0.15

In [24]:
# Save consolidated results dict
results = {
    'gsm8k': {
        'lsml':           (lsml1_auc, lsml1_lo, lsml1_hi),
        'dse':            boot_auc(labels1, -dse1),
        'lwse':           boot_auc(labels1, -lwse1),
        'sc':             boot_auc(labels1,  sc1),
        'scgpt_hard':     boot_auc(labels1, -scgpt1h),
        'scgpt_official': boot_auc(labels1, -scgpt1o),
        'fusion':         (fused1_auc, fused1_lo, fused1_hi),
        'rho_lsml_lwse':  rho1,
    },
    'math500': {
        'lsml':           (lsml2_auc, lsml2_lo, lsml2_hi),
        'dse':            boot_auc(labels2, -dse2),
        'lwse':           boot_auc(labels2, -lwse2),
        'sc':             boot_auc(labels2,  sc2),
        'scgpt_hard':     boot_auc(labels2, -scgpt2h),
        'scgpt_official': boot_auc(labels2, -scgpt2o),
        'fusion':         (fused2_auc, fused2_lo, fused2_hi),
        'rho_lsml_lwse':  rho2,
    },
    'gpqa': {
        'lsml':           (lsml3_auc, lsml3_lo, lsml3_hi),
        'vc':             (vc3_auc, vc3_lo, vc3_hi),
        'dse':            boot_auc(labels3, -dse3),
        'lwse':           boot_auc(labels3, -lwse3),
        'sc':             boot_auc(labels3,  sc3),
        'scgpt_hard':     boot_auc(labels3, -scgpt3h),
        'scgpt_official': boot_auc(labels3, -scgpt3o),
        'fusion':         (fused3_auc, fused3_lo, fused3_hi),
        'rho_lsml_lwse':  rho3,
    },
    'rag': rag_results,
}

RESULTS_PATH = os.path.join(CACHE_DIR, 'phase12_corrected_results.pkl')
_save(results, RESULTS_PATH)
print(f'Saved to {RESULTS_PATH}')

Saved to /content/drive/MyDrive/hallucination_detection/cache/phase12_corrected/phase12_corrected_results.pkl


## Step 146 — HISTORY.md Template

After the run completes, paste this into HISTORY.md with actual numbers filled in:

```
### Step 146 — Phase 12 Corrected: paper-accurate baselines + sampling fusion

**What**: Re-ran Phase 12 baselines with corrected methods:
- Likelihood-Weighted SE (LW-SE) instead of D-SE (count-only)
- SelfCheckGPT official (soft contradiction probability, premise=sentence ordering)
- L-SML continuous 1-pass (separate from K-sample loop)
- Sampling fusion: L-SML (1-pass) + LW-SE (K=10) via lsml_continuous_pipeline

**Why**: Step 145 confirmed Phase 12 baselines used D-SE and hard-argmax SelfCheckGPT.
Paper-accurate numbers needed for fair AUROC comparison in the thesis.
Advisor Item 5 (sampling fusion) implemented alongside.

**Result**: [Fill in after run]

| Domain | L-SML 1-pass | LW-SE K=10 | Fusion (L-SML+SE) | Rho |
|--------|-------------|------------|-------------------|-----|
| GSM8K  | X.XXX       | X.XXX      | X.XXX             | X.XX |
| MATH-500| X.XXX      | X.XXX      | X.XXX             | X.XX |
| GPQA   | X.XXX       | X.XXX      | X.XXX             | X.XX |

Phase 12 comparison (D-SE vs LW-SE, hard vs official SCGPT): [notes]

**Files changed**: `notebooks/Spectral_Analysis_Phase12_Corrected.ipynb`,
`cache/phase12_corrected/phase12_corrected_results.pkl`
```